In [1]:
###Cell 1: install dependencies

! pip install openai mlflow[kubernetes]

Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.4/cpu-ubi9/simple/


In [2]:
###Cell 2: hardcoded prompts

input_text = """
What's the difference between RHEL and CentOS?
"""
system_instructions = """
You are a helpful AI assistant.
You are designed to answer questions in a concise and professional manner.
"""

better_instructions = """
Answer questions with short answers, of one to three short sentences.
"""

In [3]:
###Cell 3: environment settings

import os

BASE_URL = "http://llama-32-3b-instruct-predictor.my-first-model.svc.cluster.local:8080/v1"
model_name = "llama-32-3b-instruct"

MLFLOW_TRACKING_URL = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
MLFLOW_EXPERIMENT = "simple-agent"

os.environ["MLFLOW_TRACKING_AUTH"]="kubernetes-namespaced"

In [4]:
###Cell 4: start tracing

import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URL)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.tracing.disable_notebook_display()
mlflow.openai.autolog()

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
###Cell 5: simplistic agent

from openai import OpenAI

def agent (key: str, system: str, query: str) -> str:
    client = OpenAI(
        base_url=BASE_URL,
        api_key=key
    )
    
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": query}
        ],
        max_tokens=256,
        temperature=0.7
    )

    return response.choices[0].message.content

In [6]:
###Cell 6: run the agent

# don't need this because the inferenceservice is setup with token auth disabled
#try:
#    with open("/var/run/secrets/kubernetes.io/serviceaccount/token", "r") as f:
#        AUTH_TOKEN = f.read().strip()
#except FileNotFoundError:
#    # Fallback if running outside a workbench or using a manually generated cluster token
#    AUTH_TOKEN = OCP_TOKEN

try:
    response = agent("no token", better_instructions, input_text)    
    print(response)
except Exception as e:
    print(f"An error occurred: {e}")

RHEL (Red Hat Enterprise Linux) and CentOS are both Linux distributions, but they differ in their licensing. RHEL is commercially licensed by Red Hat, while CentOS is a community-supported, free distribution based on RHEL. CentOS is often used for similar purposes as RHEL.


In [7]:
###Cell 7: deterministic scorer

from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

@scorer
def one_to_three_short_sentences(outputs: str) -> Feedback:
    """15-20 words per sentence, up to three sentences"""
    word_count = len(outputs.split())
    metric={
        "word-count": word_count,
        "min": 20,
        "max": 60
    }
    pass_fail = word_count >=20 and word_count <=60
    if word_count < metric["min"]:
        return Feedback(
                name="short_response",
                value=False,
                rationale=f"Total world count was lower than {metric["min"]}: {word_count}.",
                metadata=metric
            )
    elif word_count > metric["max"]:
        return Feedback(
                name="short_response",
                value=False,
                rationale=f"Total world count was higher than {metric["max"]}: {word_count}.",
                metadata=metric
            )
    else:
        return Feedback(
                name="short_response",
                value=True,
                rationale=f"Total world count was between {metric["min"]} and {metric["max"]}: {word_count}.",
                metadata=metric
            )

In [8]:
###Cell 8: test scorer

feedback = one_to_three_short_sentences(
    outputs="RHEL (Red Hat Enterprise Linux) is a Linux distribution."
)
print(f"Test scorer with a too short response: {feedback.value} - {feedback.rationale}")

feedback = one_to_three_short_sentences(
    outputs="RHEL (Red Hat Enterprise Linux) is a commercial Linux distribution, supported by Red Hat and produced from CentOS Stream and Fedora Linux."
)
print(f"Test scorer with an OK long response: {feedback.value} - {feedback.rationale}")

feedback = one_to_three_short_sentences(
    outputs="""
        Red Hat Enterprise Linux (RHEL) is a commercial, stable, and secure version of Linux, supported by Red Hat and produced from from multiple upstreams.
        The direct upstream to RHEL is the CentOS Stream Linux distribution, which is a free, commumity-supported distribution.
        The Fedora Linux is a direct upstream to CentOS, and thus an indirect upstream to RHEL.
        RHEL is suitable for production environments, whereas Fedora is more experimental and intended for developers and power users.
        Fedora is often used as a testing ground for new features and technologies before they are included in RHEL.
        Fedora Linux is free and commumity-supported.
    """
)
print(f"Test scorer with a too long response: {feedback.value} - {feedback.rationale}")

Test scorer with a too short response: False - Total world count was lower than 20: 9.
Test scorer with an OK long response: True - Total world count was between 20 and 60: 22.
Test scorer with a too long response: False - Total world count was higher than 60: 100.


In [9]:
###Cell 9: hardcoded evaluation data set

eval_dataset = [
    {
        "inputs": {"question":
            "What's the difference between RHEL and CentOS?"},
        "expectations": {"expected_response":
            """
            RHEL and CentOS are both Linux distributions. RHEL is a proprietary distribution from Red Hat, while CentOS is community-supported. CentOS is a direct upstream of RHEL.
            """},
    },
    {
        "inputs": {"question":
            "What's the difference between RHEL and Fedora Linux?"},
        "expectations": {"expected_response":
            """
            RHEL and Fedora are both Linux distributions. RHEL is a proprietary distribution from Red Hat, while Fedora is community-supported. Fedora is an upstream of RHEL.
            """},
    },
]

In [10]:
###Cell 10: run evaluation

def my_predict_fn(question: str) -> str:
    return agent("no token", better_instructions, input_text)

os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"]="true"

try:
    results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=my_predict_fn,
        scorers=[one_to_three_short_sentences],
    )
except Exception as e:
    print(f"An error occurred: {e}")

2026/07/31 14:35:08 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:02, Remaining: 00:00] [predict_fn: 98%, scorers: 2%]
